In [ ]:
import pandas as pd
import json
from pathlib import Path
from pdf2image import convert_from_path
from PIL import Image
import mimetypes
import re

from google import genai
from google.genai import types
from google.genai.types import HttpOptions
from google.oauth2.credentials import Credentials
from google.genai import errors as genai_errors


# ==========================
# CONFIGURATION (COMPANY VM)
# ==========================

base_url = "https://vertexai.prod.ai-gateway.quantumblack.com/7a4f3d63-b5db-4d5b-8076-8f114d1f14f7/"
access_token = "eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJhZXNKN2kxNGNidnVuTU40MTJrOU5yZ2ROeENhTlJudTNPbC1TU08ycFlJIn0.eyJleHAiOjE3NjQ5MDc0NzQsImlhdCI6MTc2NDkwNTY3NCwiYXV0aF90aW1lIjoxNzY0OTA1Njc0LCJqdGkiOiIzZDJhOTk4Yy0xNTM3LTQ5ZmUtODczYi0zNjU0ZmVkMjQwMjIiLCJpc3MiOiJodHRwczovL2F1dGgubWNraW5zZXkuaWQvYXV0aC9yZWFsbXMvciIsImF1ZCI6ImJjZDIzNzI4LTNkMjctNDQ3Yy1hMGE5LWVhY2FmMzkzYTZmNSIsInN1YiI6IjI0NDRiYzZjLTAwMzctNGIyZS1hYzI3LWZjNTlhNTkxNTM2NiIsInR5cCI6IklEIiwiYXpwIjoiYmNkMjM3MjgtM2QyNy00NDdjLWEwYTktZWFjYWYzOTNhNmY1Iiwic2Vzc2lvbl9zdGF0ZSI6IjM3ZjI2MTMyLTQyN2UtNDQyMi1iYmRmLTAxMDA0YWNmNGU1NiIsImF0X2hhc2giOiItcDdnTUlOZXJhQlQ5WW9LbElDTVNBIiwibmFtZSI6IlVnYW5kaGFyIFZhZGRpIiwiZ2l2ZW5fbmFtZSI6IlVnYW5kaGFyIiwiZmFtaWx5X25hbWUiOiJWYWRkaSIsInByZWZlcnJlZF91c2VybmFtZSI6IjE1ZDhiYmNkMmMzNTNmYWUiLCJlbWFpbCI6IlVnYW5kaGFyX1ZhZGRpQG1ja2luc2V5LmNvbSIsImFjciI6IjEiLCJzaWQiOiIzN2YyNjEzMi00MjdlLTQ0MjItYmJkZi0wMTAwNGFjZjRlNTYiLCJlbWFpbF92ZXJpZmllZCI6dHJ1ZSwiZm1ubyI6IjM0NzI3NiIsImdyb3VwcyI6WyI3YTRmM2Q2My1iNWRiLTRkNWItODA3Ni04ZjExNGQxZjE0ZjciLCJBbGwgRmlybSBVc2VycyJdfQ.g55gAN9vas_qs5pSgpvKStoxQ9CP4nlXll-2Enb-XrRJ9H5Rhw9gG42XV2vsLOJQHW9guxT5clRwwm13pHaXrdwH4oYNDjZmJ3Z9g9Kgh4zZ7-AdbVCAlr1blReRM7F9bjLnysL4ZeXxVTggd5fLZVxoSURAnWGjX-gYlLxD4azCzp1IR3nIYwvxX44qxtTENvZln2hgmpk_Z3XkBrmclawqas_JvV-DJ6ixKhrk91mS_I3DUzm2YFBwuwiedvahf0KpvTyrxU1M3U2gJvpX5XXqH7p41GSQqu0VE5-B4t3liYLIUxuqAvtkVYeN3Al3lnY1zsEPhvQOBYF0yPwVUA"
credentials = Credentials(access_token)

client = genai.Client(
    http_options=HttpOptions(
        api_version="v1",
        base_url=base_url,
    ),
    vertexai=True,
    project="aigateway",
    location="global",
    credentials=credentials,
)

MODEL_NAME = "gemini-2.5-pro"

print("Enhanced Symbol Counter Ready (High Accuracy - Anti-Hallucination).")


# ==========================
# ENHANCED ANTI-HALLUCINATION PROMPT
# ==========================

def get_enhanced_symbol_prompt():
    return """
    You are a HIGHLY PRECISE Engineering Drawing Symbol Counter AI with strong
    2D spatial understanding and STRICT adherence to accuracy over completeness.

    CORE PRINCIPLE: **ACCURACY OVER COMPLETENESS**
    - When in doubt, DO NOT COUNT
    - Under-counting is MUCH BETTER than over-counting
    - Only count what you can see with absolute certainty
    - NEVER guess, extrapolate, or assume

    INPUT:
    A single sheet of a technical drawing containing:
    1. A LEGEND/SYMBOL TABLE (usually rectangular, with rows showing symbol icons + labels)
    2. A MAIN DRAWING AREA (the actual diagram with wires, connections, and symbols)
    3. OTHER TABLES (connector tables, wire lists, BOMs - IGNORE THESE)

    YOUR TASK - THREE STRICT STEPS:

    ═══════════════════════════════════════════════════════════════
    STEP 1: LOCATE THE LEGEND TABLE (MUST BE CERTAIN)
    ═══════════════════════════════════════════════════════════════
    
    Find the table with ANY of these titles/headers:
    - "Legend"
    - "Symbol" / "Symbols"
    - "Key"
    - "Symbol Legend"
    - Similar variations
    
    CRITICAL RULES:
    ✓ The legend table has ROWS with:
      - Column 1: Small icon/symbol image
      - Column 2: Text label (e.g., "Symbol 1", "Connector", "Terminal")
      - Optional Column 3: Description/notes
    
    ✗ IGNORE these tables (they are NOT legends):
      - Connector pin tables (show pin numbers/assignments)
      - Wire lists / Cable schedules
      - BOMs / Parts lists
      - Revision history
      - Title blocks
    
    ⚠ If you cannot find a clear legend table with 100% confidence:
       Return: {"symbols": [], "legend_found": false, "message": "No legend table detected"}
       and STOP immediately.

    ═══════════════════════════════════════════════════════════════
    STEP 2: EXTRACT SYMBOL DEFINITIONS (ONLY FROM LEGEND)
    ═══════════════════════════════════════════════════════════════
    
    For EACH ROW in the legend table:
    
    - symbol_id: The exact text label from the legend row
      Examples: "Symbol 1", "Symbol 2", "Connector", "Terminal Block"
      
    - visual_description: A DETAILED description of the icon appearance
      BE SPECIFIC: shape, color, orientation, distinctive features
      Examples:
        ✓ "Small red circle with black dot in center"
        ✓ "Blue rectangle with two horizontal lines inside"
        ✓ "Green triangle pointing right with X mark"
        ✗ "Just a symbol" (too vague)
        ✗ "Icon" (not helpful)
    
    - description: Functional description if available in the legend
      (use symbol_id if no separate description exists)
    
    CRITICAL RULES:
    - ONLY extract symbols that EXIST as rows in the legend table
    - Do NOT create symbols that are not in the legend
    - Do NOT assume or infer additional symbols
    - Each symbol must have a clear icon visible in the legend

    ═══════════════════════════════════════════════════════════════
    STEP 3: COUNT IN MAIN DRAWING AREA (EXTREME CAUTION)
    ═══════════════════════════════════════════════════════════════
    
    For EACH symbol from the legend, count occurrences in the MAIN DRAWING.
    
    WHAT TO COUNT:
    ✓ Icons in the free drawing space (outside all tables)
    ✓ Symbols connected by wires/lines in the diagram
    ✓ Component symbols in the circuit/harness layout
    
    WHAT TO IGNORE (DO NOT COUNT):
    ✗ Anything inside ANY table (legend, connector tables, wire lists, etc.)
    ✗ Icons in the legend table itself
    ✗ Text labels without matching icon
    ✗ Partial or unclear symbols
    ✗ Symbols you're not 100% certain about
    
    MATCHING RULES (BE CONSERVATIVE):
    - The icon must CLEARLY match the visual_description from the legend
    - Allow minor variations:
      ✓ Rotation (0°, 90°, 180°, 270°)
      ✓ Horizontal/vertical flip
      ✓ Slight size differences (±20%)
    
    - DO NOT allow:
      ✗ Different shapes (circle vs square)
      ✗ Different colors (unless clearly the same symbol)
      ✗ Missing key features
      ✗ Ambiguous or unclear matches
    
    COUNTING DISCIPLINE:
    - Count each distinct, clear occurrence ONCE
    - If you're only 80% sure → DO NOT COUNT IT
    - If it could be two different symbols → DO NOT COUNT IT
    - If it's partially visible → DO NOT COUNT IT
    - When in doubt → count = 0
    
    PREFER ZERO OVER GUESSING:
    - count = 0 is ACCEPTABLE if you see no clear matches
    - Better to report 0 than to guess and be wrong

    ═══════════════════════════════════════════════════════════════
    STEP 4: OUTPUT FORMAT (STRICT JSON)
    ═══════════════════════════════════════════════════════════════
    
    Return ONLY this JSON structure (no markdown, no code blocks):
    
    {
      "legend_found": true,
      "legend_location": "Bottom right corner",
      "total_symbols_in_legend": 5,
      "symbols": [
        {
          "symbol_id": "Symbol 1",
          "visual_description": "Small red circle with black center dot",
          "description": "Power connection point",
          "count": 3,
          "confidence": "high"
        },
        {
          "symbol_id": "Symbol 2",
          "visual_description": "Blue square with diagonal cross",
          "description": "Ground terminal",
          "count": 0,
          "confidence": "high"
        }
      ],
      "counting_notes": "Clear legend found. Counted only unambiguous matches in main drawing area."
    }
    
    FIELD REQUIREMENTS:
    - legend_found: boolean (true/false)
    - legend_location: string describing where you found it (or null)
    - total_symbols_in_legend: integer count of legend rows
    - symbols: array (can be empty if no legend found)
    - symbol_id: string (exact from legend)
    - visual_description: string (detailed visual description)
    - description: string (functional description)
    - count: integer >= 0 (occurrences in main drawing only)
    - confidence: "high" / "medium" / "low" (your confidence in the count)
    - counting_notes: string (any observations about the counting process)
    
    ═══════════════════════════════════════════════════════════════
    FINAL REMINDERS (ANTI-HALLUCINATION PROTOCOL)
    ═══════════════════════════════════════════════════════════════
    
    1. If no legend exists → return empty symbols array
    2. Only count in main drawing area (never in tables)
    3. Only count 100% certain matches
    4. Under-counting is BETTER than over-counting
    5. Use confidence field to indicate uncertainty
    6. Provide clear counting_notes
    7. NEVER invent symbols not in the legend
    8. NEVER count ambiguous or unclear instances
    9. When doubtful → set count to 0 with "low" confidence
    10. Be honest about what you can and cannot see clearly
    """


# ==========================
# ENHANCED ANALYSIS WITH VALIDATION
# ==========================

def analyze_page_for_symbols_enhanced(image_path: str):
    """
    Enhanced analysis with strict validation and anti-hallucination measures.
    Returns structured data with confidence metrics.
    """
    print(f"   → Analyzing: {image_path}")

    with open(image_path, "rb") as f:
        image_bytes = f.read()

    mime_type, _ = mimetypes.guess_type(image_path)
    if mime_type is None:
        mime_type = "image/png"

    image_part = types.Part.from_bytes(
        data=image_bytes,
        mime_type=mime_type,
    )

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=[
                get_enhanced_symbol_prompt(),
                image_part,
            ],
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                temperature=0.1,  # Lower temperature for more consistent output
            ),
        )

        raw_text = response.text or "{}"

        # Strip code fences
        if "```json" in raw_text:
            raw_text = raw_text.split("```json")[1].split("```")[0]
        elif "```" in raw_text:
            raw_text = raw_text.split("```")[1].split("```")[0]

        data = json.loads(raw_text)

    except genai_errors.ClientError as e:
        print(f"      ✗ [ClientError] Gateway/auth issue: {getattr(e, 'message', e)}")
        return create_empty_result("API error")

    except json.JSONDecodeError as e:
        print(f"      ✗ [JSONError] Invalid JSON response: {e}")
        print(f"      Raw response: {raw_text[:200]}")
        return create_empty_result("JSON parsing error")

    except Exception as e:
        print(f"      ✗ [Error] Analysis failed: {type(e).__name__}: {e}")
        return create_empty_result(str(e))

    # Validate and normalize structure
    return validate_and_normalize(data)


def create_empty_result(reason: str):
    """Create a standardized empty result structure."""
    return {
        "legend_found": False,
        "legend_location": None,
        "total_symbols_in_legend": 0,
        "symbols": [],
        "counting_notes": f"Analysis failed or incomplete: {reason}"
    }


def validate_and_normalize(data: dict):
    """
    Validate the model's response and normalize the structure.
    Apply additional sanity checks to prevent hallucination.
    """
    if not isinstance(data, dict):
        return create_empty_result("Invalid data format")

    # Ensure required fields
    result = {
        "legend_found": data.get("legend_found", False),
        "legend_location": data.get("legend_location"),
        "total_symbols_in_legend": data.get("total_symbols_in_legend", 0),
        "symbols": [],
        "counting_notes": data.get("counting_notes", "")
    }

    symbols = data.get("symbols", [])
    if not isinstance(symbols, list):
        symbols = []

    # Validate each symbol with sanity checks
    for sym in symbols:
        if not isinstance(sym, dict):
            continue

        symbol_id = sym.get("symbol_id")
        count = sym.get("count")
        confidence = sym.get("confidence", "medium")

        # Skip if essential fields are missing
        if symbol_id is None:
            continue

        # Normalize count
        if count is None:
            count = 0
        elif not isinstance(count, int):
            try:
                count = int(count)
            except (ValueError, TypeError):
                count = 0

        # Sanity check: unreasonably high counts are likely hallucinations
        if count > 100:
            print(f"      ⚠ Warning: Suspiciously high count ({count}) for {symbol_id}, capping at 100")
            count = 100
            confidence = "low"

        # Apply confidence-based adjustments for quality control
        if confidence == "low" and count > 0:
            print(f"      ⚠ Low confidence count for {symbol_id}: {count} → treating with caution")

        result["symbols"].append({
            "symbol_id": symbol_id,
            "visual_description": sym.get("visual_description", ""),
            "description": sym.get("description", symbol_id),
            "count": count,
            "confidence": confidence
        })

    return result


# ==========================
# ENHANCED PIPELINE WITH DETAILED REPORTING
# ==========================

def process_diagram_file_enhanced(file_path: str):
    """
    Enhanced processing with detailed reporting and quality metrics.
    """
    print(f"\n{'='*70}")
    print(f"ENHANCED SYMBOL COUNTER - HIGH ACCURACY MODE")
    print(f"{'='*70}")
    print(f"📄 File: {file_path}")
    
    if not Path(file_path).exists():
        print(f"✗ ERROR: File not found: {file_path}")
        return

    ext = Path(file_path).suffix.lower()
    temp_images = []

    # Convert PDF to images
    if ext == ".pdf":
        print(f"\n🔄 Converting PDF to high-resolution images (400 DPI)...")
        try:
            pages = convert_from_path(file_path, dpi=400)
            if not pages:
                print("✗ ERROR: PDF conversion resulted in 0 pages")
                return
            
            print(f"   ✓ Successfully converted {len(pages)} pages")
            
            for i, page in enumerate(pages, start=1):
                img_path = f"temp_symbol_page_{i}.png"
                page.save(img_path, "PNG")
                temp_images.append(img_path)
                
        except Exception as e:
            print(f"✗ ERROR: PDF conversion failed: {e}")
            return
    else:
        temp_images = [file_path]
        print(f"   ✓ Using image file directly")

    # Data collection structures
    total_counts = {}  # (symbol_id) -> {"count": int, "confidence": str, "description": str}
    page_rows = []
    legend_info = []

    # Process each page
    print(f"\n{'─'*70}")
    print(f"ANALYZING PAGES")
    print(f"{'─'*70}")

    for page_idx, img in enumerate(temp_images, start=1):
        print(f"\n📊 Page {page_idx}/{len(temp_images)}")
        result = analyze_page_for_symbols_enhanced(img)

        legend_found = result.get("legend_found", False)
        legend_location = result.get("legend_location", "N/A")
        total_in_legend = result.get("total_symbols_in_legend", 0)
        counting_notes = result.get("counting_notes", "")
        symbols = result.get("symbols", [])

        # Record legend information
        if legend_found:
            print(f"   ✓ Legend found: {legend_location}")
            print(f"   ✓ Symbols in legend: {total_in_legend}")
            legend_info.append({
                "Page": page_idx,
                "Legend_Found": "Yes",
                "Location": legend_location,
                "Symbols_In_Legend": total_in_legend,
                "Notes": counting_notes
            })
        else:
            print(f"   ✗ No legend table detected")
            legend_info.append({
                "Page": page_idx,
                "Legend_Found": "No",
                "Location": "N/A",
                "Symbols_In_Legend": 0,
                "Notes": counting_notes
            })

        if not symbols:
            print(f"   → No symbols extracted from this page")
            continue

        print(f"   → Processing {len(symbols)} symbols:")

        for sym in symbols:
            symbol_id = sym.get("symbol_id")
            count = sym.get("count", 0)
            confidence = sym.get("confidence", "medium")
            description = sym.get("description", "")
            visual_desc = sym.get("visual_description", "")

            # Aggregate totals
            if symbol_id not in total_counts:
                total_counts[symbol_id] = {
                    "count": 0,
                    "confidence": confidence,
                    "description": description,
                    "visual_description": visual_desc,
                    "pages_found": []
                }
            
            total_counts[symbol_id]["count"] += count
            if count > 0:
                total_counts[symbol_id]["pages_found"].append(page_idx)

            # Record page-level data
            page_rows.append({
                "Page": page_idx,
                "Symbol_ID": symbol_id,
                "Visual_Description": visual_desc,
                "Description": description,
                "Count_On_Page": count,
                "Confidence": confidence
            })

            # Display
            conf_icon = {"high": "✓", "medium": "~", "low": "?"}.get(confidence, "?")
            print(f"      {conf_icon} {symbol_id}: {count} occurrences ({confidence} confidence)")

    # Generate summary
    print(f"\n{'='*70}")
    print(f"RESULTS SUMMARY")
    print(f"{'='*70}")
    print(f"Pages processed: {len(temp_images)}")
    print(f"Pages with legend: {sum(1 for l in legend_info if l['Legend_Found'] == 'Yes')}")
    print(f"Unique symbols found: {len(total_counts)}")
    print(f"Total symbol instances: {sum(v['count'] for v in total_counts.values())}")

    if not total_counts:
        print(f"\n⚠ No symbols counted. Excel file will not be created.")
        print(f"   Possible reasons:")
        print(f"   - No legend table found in the drawing")
        print(f"   - Legend exists but no symbols in main drawing area")
        print(f"   - Image quality too low for accurate detection")
        # Cleanup
        if ext == ".pdf":
            for img in temp_images:
                Path(img).unlink(missing_ok=True)
        return

    # Create detailed Excel output
    print(f"\n📝 Creating Excel report...")

    output_filename = f"{Path(file_path).stem}_SYMBOL_COUNTS_ENHANCED.xlsx"

    # Summary sheet
    summary_rows = []
    for symbol_id, data in total_counts.items():
        summary_rows.append({
            "Symbol_ID": symbol_id,
            "Visual_Description": data["visual_description"],
            "Description": data["description"],
            "Total_Count": data["count"],
            "Confidence": data["confidence"],
            "Found_On_Pages": ", ".join(map(str, data["pages_found"])) if data["pages_found"] else "None"
        })

    df_summary = pd.DataFrame(summary_rows)
    df_by_page = pd.DataFrame(page_rows)
    df_legend = pd.DataFrame(legend_info)

    with pd.ExcelWriter(output_filename, engine="openpyxl") as writer:
        df_summary.to_excel(writer, sheet_name="Summary", index=False)
        df_by_page.to_excel(writer, sheet_name="By_Page", index=False)
        df_legend.to_excel(writer, sheet_name="Legend_Info", index=False)

    print(f"   ✓ Summary sheet: {len(summary_rows)} symbols")
    print(f"   ✓ By_Page sheet: {len(page_rows)} records")
    print(f"   ✓ Legend_Info sheet: {len(legend_info)} pages")
    print(f"\n✅ SUCCESS: Excel report created → {output_filename}")

    # Quality metrics
    print(f"\n📊 QUALITY METRICS:")
    high_conf = sum(1 for d in total_counts.values() if d["confidence"] == "high")
    med_conf = sum(1 for d in total_counts.values() if d["confidence"] == "medium")
    low_conf = sum(1 for d in total_counts.values() if d["confidence"] == "low")
    
    print(f"   High confidence: {high_conf}/{len(total_counts)} symbols")
    print(f"   Medium confidence: {med_conf}/{len(total_counts)} symbols")
    print(f"   Low confidence: {low_conf}/{len(total_counts)} symbols")
    
    if low_conf > 0:
        print(f"\n   ⚠ Note: {low_conf} symbol(s) have low confidence counts.")
        print(f"      Consider manual verification for these symbols.")

    # Cleanup temp images
    if ext == ".pdf":
        for img in temp_images:
            Path(img).unlink(missing_ok=True)
        print(f"\n🧹 Cleaned up temporary files")


# ==========================
# ENTRY POINT
# ==========================

if __name__ == "__main__":
    print("\n" + "="*70)
    print("ENHANCED SYMBOL COUNTER")
    print("High Accuracy Mode with Anti-Hallucination Protocol")
    print("="*70)
    print("\nFeatures:")
    print("  ✓ Strict legend-only symbol extraction")
    print("  ✓ Conservative counting (accuracy over completeness)")
    print("  ✓ Confidence metrics for each count")
    print("  ✓ Detailed visual descriptions")
    print("  ✓ Quality validation and sanity checks")
    print("  ✓ Comprehensive Excel reporting")
    
    print("\nReady to process files.")
    print("\nUsage:")
    print("  process_diagram_file_enhanced('your_drawing.pdf')")
    print("\nExample:")
    print("  process_diagram_file_enhanced('harness_diagram.pdf')")

Enhanced Symbol Counter Ready (High Accuracy - Anti-Hallucination).

ENHANCED SYMBOL COUNTER
High Accuracy Mode with Anti-Hallucination Protocol

Features:
  ✓ Strict legend-only symbol extraction
  ✓ Conservative counting (accuracy over completeness)
  ✓ Confidence metrics for each count
  ✓ Detailed visual descriptions
  ✓ Quality validation and sanity checks
  ✓ Comprehensive Excel reporting

Ready to process files.

Usage:
  process_diagram_file_enhanced('your_drawing.pdf')

Example:
  process_diagram_file_enhanced('harness_diagram.pdf')


In [ ]:
process_diagram_file_enhanced('Master test.png')